# V98 - Ridge bagging

Pre-registered plan: blends across views failed, but bagging across bootstrap samples was never tested. Ten Ridge refits on bootstrap resamples with out-of-bag predictions, so every user is scored only by models that never saw them. Same features and grouped structure under a fresh seed, gated against single Ridge with the usual locks.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2424
N_SPLITS = 5
GATE_FLOOR = 0.002
BAGS = 10
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v98")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Out-of-bag bagged predictions, then gate

Each user averages only bags that excluded them. Users missed by no bag fall back to single Ridge, counted openly.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_single = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_single[va, j] = m.predict(W[va])
    fids[va] = fi + 1
acc = np.zeros_like(Y)
cnt = np.zeros(n_train)
for b in range(BAGS):
    idx = rng.choice(n_train, n_train, replace=True)
    oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[idx], Y[idx, j])
        acc[oob, j] += m.predict(W[oob])
    cnt[oob] += 1
oof_bag = np.where(cnt[:, None] > 0, acc / np.maximum(cnt[:, None], 1), oof_single)
print(f"users with zero OOB bags: {(cnt == 0).sum()}")
base = ndcg5(Y, oof_single)
d = ndcg5(Y, oof_bag) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_bag[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"single={base.mean():.5f} bagged={ndcg5(Y, oof_bag).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT bagging" if (lo > GATE_FLOOR and wf >= 0) else "KEEP single fit; bagging adds nothing")

users with zero OOB bags: 45
single=0.62517 bagged=0.64194 delta=+0.01677 CI95=[+0.01323,+0.02022] worst-fold=+0.00136
ADOPT bagging
